# Haya Skin Analyzer — YOLOv8 Training v2

## ⚠️ READ THIS BEFORE RUNNING ⚠️

**DO NOT run cells one by one (interactive mode).**
If you do, the session will expire and all files will be lost.

### Correct way to run:
1. Make sure Accelerator = **GPU T4 x1** (Settings panel on the right)
2. Click **Save Version** button (top right corner)
3. Choose **Save & Run All (Commit)**
4. Click **Save**
5. Wait ~1 hour — Kaggle runs everything automatically
6. When it says **Complete**, click **Output** tab on the right
7. Download **best.pt**

That's it. Do NOT touch anything while it runs.

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()
print('\nDependencies ready.')

In [ ]:
# Cell 2 — Config (do not change anything here)
ROBOFLOW_API_KEY = "Dej9oDfQTdZXUPl6xeQe"
EPOCHS     = 60
MODEL_SIZE = "yolov8n"
IMG_SIZE   = 416
BATCH      = 32
OUTPUT_DIR = "/kaggle/working"
print('Config ready.')

In [ ]:
# Cell 3 — Download datasets, fix class names, merge
from roboflow import Roboflow
import os, shutil, yaml, re
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

datasets_to_download = [
    ("xzesoxee",                 "acne-detection-9z3qf", 3),   # Acne
    ("skin-condition-detection",  "dark-circles-19mqw",   1),   # Dark Circles
    ("wrinkle-old-bucket",        "wrinkle-rkcym",       17),   # Wrinkles
]

# Clean canonical class names the backend expects
CANONICAL = ["acne", "dark_circles", "wrinkles", "pigmentation"]

def normalize_class(name: str):
    """Map any messy dataset class name to our canonical name, or None to drop it."""
    n = str(name).lower().strip()
    if any(k in n for k in ["acne", "pimple", "spot", "blackhead", "whitehead", "comedone", "blemish"]):
        return "acne"
    if any(k in n for k in ["dark", "circle", "puffy", "eye bag", "eyebag", "periorbital"]):
        return "dark_circles"
    if any(k in n for k in ["wrinkle", "fine line", "fineline", "crow", "forehead line"]):
        return "wrinkles"
    if any(k in n for k in ["pigment", "melasma", "freckle", "dark spot", "hyperpig"]):
        return "pigmentation"
    if re.fullmatch(r'\d+', n):   # pure number like '0' — unknown, drop
        return None
    return None

# Download
downloaded = []
for workspace, project_name, version_num in datasets_to_download:
    try:
        print(f"Downloading {project_name}...")
        ds = rf.workspace(workspace).project(project_name).version(version_num).download("yolov8")
        downloaded.append(ds.location)
        print(f"  OK")
    except Exception as e:
        print(f"  FAILED: {e}")

if not downloaded:
    raise RuntimeError("No datasets downloaded — check API key.")

print(f"\nDownloaded {len(downloaded)}/3 datasets")

# Merge with class name fixing
MERGED = Path(f"{OUTPUT_DIR}/merged")
for split in ("train", "valid", "test"):
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

for ds_path in downloaded:
    ds_path = Path(ds_path)
    yaml_file = next(ds_path.glob("*.yaml"), None)
    if not yaml_file:
        continue
    with open(yaml_file) as f:
        dy = yaml.safe_load(f)
    raw_names = dy.get("names", [])

    # Build remap: old_class_id -> new canonical id
    id_remap = {}
    for old_id, raw_name in enumerate(raw_names):
        canonical = normalize_class(raw_name)
        if canonical:
            id_remap[old_id] = CANONICAL.index(canonical)
    print(f"  {ds_path.name}: {raw_names} -> {id_remap}")

    for split in ("train", "valid", "test"):
        # Copy images
        for img in (ds_path / split / "images").glob("*"):
            shutil.copy(img, MERGED / split / "images" / img.name)
        # Rewrite labels with corrected class ids
        for lbl in (ds_path / split / "labels").glob("*.txt"):
            new_lines = []
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    old_id = int(float(parts[0]))
                    if old_id in id_remap:
                        parts[0] = str(id_remap[old_id])
                        new_lines.append(" ".join(parts))
            if new_lines:
                with open(MERGED / split / "labels" / lbl.name, "w") as f:
                    f.write("\n".join(new_lines))

# Write data.yaml
DATA_YAML = str(MERGED / "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.dump({
        "path":  str(MERGED),
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(CANONICAL),
        "names": CANONICAL,
    }, f)

print(f"\nFinal classes: {CANONICAL}")
for split in ("train", "valid", "test"):
    n = len(list((MERGED / split / "images").glob("*")))
    print(f"  {split}: {n} images")
print("\nDataset ready.")

In [ ]:
# Cell 4 — Train (~1 hour on T4 GPU)
from ultralytics import YOLO

model = YOLO(f"{MODEL_SIZE}.pt")

results = model.train(
    data         = DATA_YAML,
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH,
    patience     = 20,
    save         = True,
    save_period  = 5,
    project      = OUTPUT_DIR,
    name         = "haya_skin_v2",
    exist_ok     = True,
    device       = 0,
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs= 3,
    weight_decay = 0.0005,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.1,
    fliplr       = 0.5,
    degrees      = 10.0,
    hsv_s        = 0.5,
    hsv_v        = 0.3,
    val          = True,
    plots        = True,
    verbose      = False,
)

print("Training complete.")

In [ ]:
# Cell 5 — Save best.pt to Kaggle output (runs automatically in commit mode)
import shutil, glob, os
from ultralytics import YOLO

print("Searching for best.pt...")
candidates = glob.glob(f"{OUTPUT_DIR}/**/best.pt", recursive=True)
if not candidates:
    print("best.pt not found, trying last.pt...")
    candidates = glob.glob(f"{OUTPUT_DIR}/**/last.pt", recursive=True)
if not candidates:
    raise FileNotFoundError("No model file found. Training likely crashed. Check Cell 4 logs.")

source = candidates[0]
dest   = f"{OUTPUT_DIR}/best.pt"

print(f"Found: {source}")
shutil.copy(source, dest)

size_mb = os.path.getsize(dest) / 1024 / 1024
print(f"Copied to: {dest} ({size_mb:.1f} MB)")

if size_mb < 1:
    raise RuntimeError(f"File is only {size_mb:.2f} MB — training likely failed.")

# Validate and show per-class results
print("\nValidating model...")
m = YOLO(dest)
metrics = m.val(data=DATA_YAML, verbose=False)

print(f"\n{'='*40}")
print(f"Overall mAP50 : {metrics.box.map50:.3f}")
print(f"Precision     : {metrics.box.mp:.3f}")
print(f"Recall        : {metrics.box.mr:.3f}")
print(f"\nPer-class mAP50:")
for name, ap in zip(m.names.values(), metrics.box.ap50):
    bar = '█' * int(ap * 20)
    print(f"  {name:<15} {ap:.3f}  {bar}")
print(f"{'='*40}")
print(f"\nClasses: {list(m.names.values())}")
print(f"\nbest.pt is at: {dest}")
print("It will appear in the Output tab once the commit run finishes.")